In [1]:
from IPython.display import display, Markdown, IFrame
from ipywidgets import Button, Layout, GridBox, VBox, Output
from pathlib import Path
import rospy
from sidecar import Sidecar
from helper import  Robots, Rvizweb
import subprocess

RVIZWEB_URL = url = rospy.get_param('/rvizweb/jupyter_proxy_url')
IFRAME = IFrame(src=RVIZWEB_URL, width='100%', height='100%')
def MAKE_SIDECAR():
    return Sidecar(title='RVIZWEB', anchor='right')
RVIZWEB = Rvizweb(make_sidecar=MAKE_SIDECAR, iframe=IFRAME, display=display)
ROBOTS = Robots()

def check_file(file: Path | str, ending: str = "", parameter_name: str = "") -> Path:
    message_prefix = (
        f"Parameter {parameter_name}" if parameter_name != "" else "Given File"
    )
    try:
        file = Path(file)
    except:
        raise ValueError(f"{message_prefix} must be a valid file path")
    if not file.is_file():
        raise ValueError(f"{message_prefix} must be pointing to an existing file")
    if ending != "" and not file.match("*" + ending):
        raise ValueError(f"{message_prefix} must be pointing to an {ending} file")
    return file
    
class Launcher2:
    COMMAND_TEMPLATE = "roslaunch {launch_file}{args}"

    def __init__(self, launch_file: Path | str):
        launch_file = check_file(
            file=launch_file,
            ending=".launch",
            parameter_name="launch_file"
        )

        self.launch_file = launch_file
        self.open_process: subprocess.Popen | None = None
        self.process_name: str | None = None

    def _build_command(self, urdf_file: Path | str | None):
        # No urdf_file -> plain roslaunch
        if urdf_file is None:
            args = ""
        else:
            args = f" urdf_file:={urdf_file}"

        return self.COMMAND_TEMPLATE.format(
            launch_file=self.launch_file,
            args=args
        )

    def launch(
        self,
        rvizweb,
        process_name: str | None = None,
        ferr=None,
        fout=None,
    ):
        # stop previous process
        if self.open_process:
            self.open_process.kill()

        cmd = self._build_command(None)

        self.open_process = subprocess.Popen(
            ["/bin/bash", "-c", cmd],
            stdout=fout if fout else subprocess.DEVNULL,
            stderr=ferr if ferr else subprocess.DEVNULL,
            shell=False,
        )

        self.process_name = process_name or "Unknown"
        rvizweb.open()

    def kill(self):
        if self.open_process:
            self.open_process.kill()
        self.open_process = None
        self.process_name = None

    def get_process_name(self) -> str | None:
        return self.process_name

LAUNCHER2 = Launcher2(Path.home() /'work/launch/test.launch')


In [2]:
def create_button():
    name = "test"
    btn = Button(
        description=name,
        layout=Layout(width='auto', height='50px'),
        style={'font_size':'1rem'},
        tooltip=f"Launch robot: {name}"
    )
    def launch_robot():
        
        urdf_file = ROBOTS.get_urdf_file(name)
        if urdf_file:
            print(f'Launch robot {name} with urdf file {str(urdf_file)}')
            LAUNCHER2.launch(urdf_file=none, 
                            rvizweb=RVIZWEB)
        else:
            print(f'No urdf file for robot  available')
            LAUNCHER2.launch( 
                            rvizweb=RVIZWEB)
    btn.on_click(lambda b: launch_robot())
    return btn

if LAUNCHER2.get_process_name() != None:
    display(Markdown(f'### Robot **{LAUNCHER2.get_process_name()}** Launched'))
display(Markdown('#### Click the buttons below to launch an industrial robot.'))
display(VBox([create_button()]))

#### Click the buttons below to launch an industrial robot.

### 
### ROS Script: 

In [ ]:
import rospy
import moveit_commander
from geometry_msgs.msg import Pose

# ---- safe ROS init (ONLY ONCE) ----
if not rospy.core.is_initialized():
    rospy.init_node("ur5_python_moveit", anonymous=True)

# ---- MoveIt init (ONLY ONCE per kernel) ----
moveit_commander.roscpp_initialize([])

# ---- create robot + planning interface ----
robot = moveit_commander.RobotCommander()
scene = moveit_commander.PlanningSceneInterface()

group = moveit_commander.MoveGroupCommander("manipulator")

# ---- planning settings ----
group.set_planning_time(5)
group.set_num_planning_attempts(10)
group.allow_replanning(True)

print("✅ MoveIt initialized")
print("Available groups:", robot.get_group_names())

In [ ]:
group.set_named_target("home")
group.go(wait=True)
group.stop()
group.clear_pose_targets()

print(" Moved to home")

In [ ]:
pose = Pose()
pose.position.x = 0.4
pose.position.y = 0.2
pose.position.z = 0.3
pose.orientation.w = 1.0

group.set_pose_target(pose)
group.go(wait=True)
group.stop()
group.clear_pose_targets()

print(" Moved to Target A")

In [ ]:
pose = Pose()
pose.position.x = 0.3
pose.position.y = -0.2
pose.position.z = 0.5
pose.orientation.w = 1.0

group.set_pose_target(pose)
group.go(wait=True)
group.stop()
group.clear_pose_targets()

print(" Moved to Target B")